# 05 — Train and evaluate dummy baselines

This notebook fits the mean and median dummy regressors on the completed human CB2 modeling preparation handoff. Each dummy is fitted separately for the random and scaffold split protocols, then predicts training and validation pKi. The saved models, predictions, and metrics belong to this notebook; test subsets remain reserved.

The final sections add a bounded training-only CV search and save its selected models, predictions, and baseline comparison in a separate tuning subfolder.

**Reference:** All source and supporting files are in [provenance](../provenance/README.md). The data provenance [Excel workbook](../provenance/CB2_data_provenance.xlsx) is the supplied presentation snapshot; reruns preserve it unchanged. Saved outputs below describe the previous execution; printed old paths can differ. Rerun notebooks 00–09 in order to refresh the complete pipeline.


## 1. Verify one completed preparation run

The preparation manifest fixes the curation source, fingerprint arrays, and split assignments. This cell checks its completion record, input hashes, and relevant software versions before model fitting. The exact run path is explicit so a later folder cannot be selected by accident.

In [1]:
from pathlib import Path
import sys
# Shared storage helpers keep the notebooks on the same completed dataset.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "scripts"))
from provenance_support import (
    latest_acquisition, current_selection, current_curation, current_preparation,
    current_mlp_baseline, save_review_record, load_review_record,
)
from datetime import datetime, timezone
from time import perf_counter
import hashlib
import json
import platform

import joblib
import numpy as np
import pandas as pd
import rdkit
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PREP_MANIFEST_REL = current_preparation(project_root)
STRATEGIES = ("random", "scaffold")
SUBSETS = ("train", "validation", "test")
DUMMY_STRATEGIES = ("mean", "median")
run_started_at_utc = datetime.now(timezone.utc).isoformat()
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
assert (project_root / "notebooks/03_modeling_preparation.ipynb").is_file()

def sha256_file(path):
    """Hash a file without loading all its bytes into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Require the complete preparation handoff before any target-dependent fit.
prep_manifest_path = project_root / PREP_MANIFEST_REL
assert prep_manifest_path.is_file(), f"Preparation manifest is missing: {prep_manifest_path}"
with prep_manifest_path.open(encoding="utf-8") as stream:
    prep_manifest = json.load(stream)
assert prep_manifest.get("completed_at_utc") and prep_manifest.get("stage") == "modeling_preparation"
for check in (
    "source_hashes", "saved_arrays_and_fingerprint_bits", "saved_splits_and_scaffolds",
    "saved_audit_reports",
):
    assert prep_manifest.get("verification", {}).get(check) is True, check
assert prep_manifest.get("models_fitted") is False
prep_folder_rel = Path(prep_manifest["output_folder"])
assert (project_root / prep_folder_rel).resolve() == prep_manifest_path.parent.resolve()
required_filenames = ("fingerprints_and_targets.npz", "split_assignments.csv")
artifact_paths = {}
for filename in required_filenames:
    relative_name = (prep_folder_rel / filename).as_posix()
    assert relative_name in prep_manifest["artifact_sha256"], filename
    artifact_path = project_root / relative_name
    assert artifact_path.is_file(), f"Preparation artifact is missing: {relative_name}"
    assert sha256_file(artifact_path) == prep_manifest["artifact_sha256"][relative_name], relative_name
    artifact_paths[filename] = artifact_path
source_manifest_hash = sha256_file(prep_manifest_path)

# Check versions before loading arrays and training scikit-learn estimators.
installed_versions = {
    "python": platform.python_version(), "rdkit": rdkit.__version__,
    "numpy": np.__version__, "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__, "joblib": joblib.__version__,
}
version_differences = {
    package: (saved, installed_versions.get(package))
    for package, saved in prep_manifest["software_versions"].items()
    if installed_versions.get(package) != saved
}
assert not version_differences, f"Preparation/current software versions differ: {version_differences}"
print(f"{'Preparation run':<27} | {PREP_MANIFEST_REL.as_posix()}")
print(f"{'Verified input artifacts':<27} | {len(artifact_paths):>7,}")
print(f"{'scikit-learn version':<27} | {sklearn.__version__}")

Preparation run             | data/processed/modeling_preparation/run_01/manifest.json
Verified input artifacts    |       2
scikit-learn version        | 1.9.0


## 2. Align the saved arrays and subsets

The NPZ contains fixed fingerprint bits, one target per curated structure, and structure keys in the same row order. Split labels are mapped by key instead of assuming the assignment CSV has that order. The test rows remain identified so neither dummy can train on or predict them here.

In [2]:
# NumPy pickle loading is disabled because the preparation arrays contain no objects.
with np.load(artifact_paths["fingerprints_and_targets.npz"], allow_pickle=False) as saved:
    X = saved["X"]
    y = saved["y"]
    structure_keys = saved["rdkit_smiles"]
feature_width = int(prep_manifest["configuration"]["fingerprint"]["bits"])
assert X.ndim == 2 and X.shape[1] == feature_width
assert X.dtype == np.uint8 and np.isin(X, (0, 1)).all()
assert y.ndim == 1 and len(X) == len(y) == len(structure_keys)
assert np.isfinite(y).all() and structure_keys.dtype.kind == "U"
assert len(set(structure_keys)) == len(structure_keys) == prep_manifest["counts"]["structures"]

# One assignment per key per strategy is necessary for reliable mask alignment.
assignments = pd.read_csv(artifact_paths["split_assignments.csv"], dtype=str, keep_default_na=False)
assert list(assignments.columns) == [
    "split_strategy", "rdkit_smiles", "subset", "scaffold", "fingerprint_group_id"
]
assert len(assignments) == len(structure_keys) * len(STRATEGIES)
assert not assignments.duplicated(["split_strategy", "rdkit_smiles"]).any()
assert set(assignments["split_strategy"]) == set(STRATEGIES)
assert set(assignments["subset"]) == set(SUBSETS)
split_indices = {}
for strategy in STRATEGIES:
    rows = assignments.loc[assignments["split_strategy"].eq(strategy)]
    assert set(rows["rdkit_smiles"]) == set(structure_keys)
    subset_by_key = dict(zip(rows["rdkit_smiles"], rows["subset"]))
    labels = np.asarray([subset_by_key[key] for key in structure_keys])
    split_indices[strategy] = {}
    for subset in SUBSETS:
        indices = np.flatnonzero(labels == subset)
        expected = prep_manifest["counts"]["split_subsets"][strategy][subset]["structures"]
        assert len(indices) == expected and len(indices) > 0
        assert set(structure_keys[indices]) == set(rows.loc[rows["subset"].eq(subset), "rdkit_smiles"])
        split_indices[strategy][subset] = indices
    assert set(split_indices[strategy]["train"]).isdisjoint(split_indices[strategy]["validation"])
    assert set(split_indices[strategy]["train"]).isdisjoint(split_indices[strategy]["test"])
    assert set(split_indices[strategy]["validation"]).isdisjoint(split_indices[strategy]["test"])
    print(f"{strategy:<10} | train {len(split_indices[strategy]['train']):>5,} | "
          f"validation {len(split_indices[strategy]['validation']):>4,} | "
          f"reserved test {len(split_indices[strategy]['test']):>4,}")

random     | train 2,860 | validation  358 | reserved test  358
scaffold   | train 2,778 | validation  461 | reserved test  337


## 3. Fit mean and median dummies

A dummy ignores molecular fingerprint patterns and predicts a constant learned from its training labels. Mean and median are different constants, and each split protocol has a different training set. Four fresh estimators are fitted below; neither validation nor test labels enter those fitted constants.

In [3]:
models = {}
training_rows = []
training_slices = {}
independent_constants = {}
for strategy in STRATEGIES:
    train_indices = split_indices[strategy]["train"]
    assert len(X[train_indices]) == len(y[train_indices])
    training_slices[strategy] = train_indices[:5]
    for dummy_strategy in DUMMY_STRATEGIES:
        model_name = f"{strategy}_dummy_{dummy_strategy}"
        # Calculate the expected constant without reading it from the estimator.
        expected = float(np.mean(y[train_indices]) if dummy_strategy == "mean"
                         else np.median(y[train_indices]))
        started = perf_counter()
        model = DummyRegressor(strategy=dummy_strategy)
        model.fit(X[train_indices], y[train_indices])
        fit_seconds = perf_counter() - started
        assert model.n_features_in_ == feature_width
        np.testing.assert_allclose(model.predict(X[training_slices[strategy]]), expected,
                                   rtol=0, atol=1e-10)
        models[model_name] = model
        independent_constants[model_name] = expected
        training_rows.append({
            "split_strategy": strategy, "model": model_name,
            "dummy_strategy": dummy_strategy, "training_structures": len(train_indices),
            "features": feature_width, "fit_seconds": fit_seconds,
            "training_constant_pki": expected,
        })
        print(f"{model_name:<26} | {len(train_indices):>5,} training structures | "
              f"constant {expected:.6f} pKi")
training_summary = pd.DataFrame(training_rows)
assert len(models) == 4

random_dummy_mean          | 2,860 training structures | constant 7.096994 pKi
random_dummy_median        | 2,860 training structures | constant 7.070581 pKi
scaffold_dummy_mean        | 2,778 training structures | constant 7.063788 pKi
scaffold_dummy_median      | 2,778 training structures | constant 7.036212 pKi


## 4. Predict on training and validation rows

Each fitted dummy predicts every training and validation structure in its own protocol. The residual is predicted minus observed pKi, making positive residuals overpredictions of binding affinity on this scale. The model and split strategy remain explicit in every exported row; test rows are not passed to `predict`.

In [4]:
prediction_rows = []
for strategy in STRATEGIES:
    for dummy_strategy in DUMMY_STRATEGIES:
        model_name = f"{strategy}_dummy_{dummy_strategy}"
        model = models[model_name]
        for subset in ("train", "validation"):
            indices = split_indices[strategy][subset]
            predicted = model.predict(X[indices])
            assert len(predicted) == len(indices) and np.isfinite(predicted).all()
            # The model name matters because mean and median share each subset's keys.
            for index, predicted_pki in zip(indices, predicted):
                observed_pki = float(y[index])
                residual = float(predicted_pki - observed_pki)
                prediction_rows.append({
                    "split_strategy": strategy, "model": model_name, "subset": subset,
                    "rdkit_smiles": str(structure_keys[index]),
                    "observed_pki": observed_pki, "predicted_pki": float(predicted_pki),
                    "residual": residual, "absolute_error": abs(residual),
                    "squared_error": residual ** 2,
                })
predictions = pd.DataFrame(prediction_rows)
predictions = predictions.sort_values(
    ["split_strategy", "model", "subset", "rdkit_smiles"]
).reset_index(drop=True)
assert not predictions.duplicated(["split_strategy", "model", "subset", "rdkit_smiles"]).any()
assert set(predictions["subset"]) == {"train", "validation"}
for strategy in STRATEGIES:
    for dummy_strategy in DUMMY_STRATEGIES:
        model_name = f"{strategy}_dummy_{dummy_strategy}"
        for subset in ("train", "validation"):
            rows = predictions.loc[
                predictions["split_strategy"].eq(strategy)
                & predictions["model"].eq(model_name)
                & predictions["subset"].eq(subset)
            ]
            assert set(rows["rdkit_smiles"]) == set(structure_keys[split_indices[strategy][subset]])
print(f"Generated {len(predictions):,} dummy training and validation prediction rows; no test rows.")

Generated 12,914 dummy training and validation prediction rows; no test rows.


### Inspect a few dummy predictions

The table shows one training and one validation example for each fitted dummy and protocol. Dummies predict their learned training constant for every row in a subset; observed values and residuals vary by structure. The saved CSV contains all prediction rows with full keys and precision.


In [5]:
# Show one row per subset for every fitted dummy without overwhelming the notebook.
print(f"{'Protocol':<10} | {'Dummy':<7} | {'Subset':<10} | {'Structure key (shortened)':<35} | {'Observed':>8} | {'Predicted':>9} | {'Residual':>8}")
for strategy in STRATEGIES:
    for dummy_strategy in DUMMY_STRATEGIES:
        model_name = f"{strategy}_dummy_{dummy_strategy}"
        for subset in ("train", "validation"):
            row = predictions.loc[
                predictions["split_strategy"].eq(strategy)
                & predictions["model"].eq(model_name)
                & predictions["subset"].eq(subset)
            ].iloc[0]
            full_key = row["rdkit_smiles"]
            key = full_key if len(full_key) <= 35 else full_key[:34] + "…"
            print(f"{strategy:<10} | {dummy_strategy:<7} | {subset:<10} | {key:<35} | "
                  f"{row['observed_pki']:>8.3f} | {row['predicted_pki']:>9.3f} | {row['residual']:>+8.3f}")


Protocol   | Dummy   | Subset     | Structure key (shortened)           | Observed | Predicted | Residual
random     | mean    | train      | C#CCCC1(CCNC(=O)c2nonc2Cn2nc3nc(C(… |    6.031 |     7.097 |   +1.066
random     | mean    | validation | Brc1ccc2c(c1)c(OCc1ccc3ccccc3c1)nn… |    5.745 |     7.097 |   +1.352
random     | median  | train      | C#CCCC1(CCNC(=O)c2nonc2Cn2nc3nc(C(… |    6.031 |     7.071 |   +1.040
random     | median  | validation | Brc1ccc2c(c1)c(OCc1ccc3ccccc3c1)nn… |    5.745 |     7.071 |   +1.326
scaffold   | mean    | train      | C#CCCC1(CCNC(=O)c2nonc2Cn2nc3nc(C(… |    6.031 |     7.064 |   +1.033
scaffold   | mean    | validation | C#Cc1cccc(S(=O)(=O)F)c1Cn1nc2nc(C(… |    6.376 |     7.064 |   +0.688
scaffold   | median  | train      | C#CCCC1(CCNC(=O)c2nonc2Cn2nc3nc(C(… |    6.031 |     7.036 |   +1.006
scaffold   | median  | validation | C#Cc1cccc(S(=O)(=O)F)c1Cn1nc2nc(C(… |    6.376 |     7.036 |   +0.660


## 5. Calculate baseline metrics

MAE is average absolute pKi error, and RMSE places greater weight on large errors. R² compares squared error with predicting the evaluated subset's mean; it can be negative on validation rows and is undefined for fewer than two observations or a constant target. These values describe dummies only; forest comparisons can use the separately verified model reports later.

In [6]:
metric_rows = []
for strategy in STRATEGIES:
    for dummy_strategy in DUMMY_STRATEGIES:
        model_name = f"{strategy}_dummy_{dummy_strategy}"
        for subset in ("train", "validation"):
            rows = predictions.loc[
                predictions["split_strategy"].eq(strategy)
                & predictions["model"].eq(model_name)
                & predictions["subset"].eq(subset)
            ]
            observed = rows["observed_pki"].to_numpy()
            predicted = rows["predicted_pki"].to_numpy()
            r2_defined = len(rows) >= 2 and np.max(observed) > np.min(observed)
            metric_rows.append({
                "split_strategy": strategy, "model": model_name, "subset": subset,
                "n_structures": len(rows),
                "mae_pki": float(mean_absolute_error(observed, predicted)),
                "rmse_pki": float(np.sqrt(mean_squared_error(observed, predicted))),
                "r2": float(r2_score(observed, predicted)) if r2_defined else np.nan,
                "r2_status": "defined" if r2_defined else "undefined_small_or_constant_target",
            })
metrics = pd.DataFrame(metric_rows)
print(f"{'Protocol':<10} | {'Dummy':<7} | {'Subset':<10} | {'N':>5} | {'MAE':>7} | {'RMSE':>7} | {'R²':>7}")
for row in metrics.itertuples(index=False):
    r2_text = f"{row.r2:.3f}" if row.r2_status == "defined" else "undefined"
    dummy_strategy = row.model.rsplit("_", 1)[-1]
    print(f"{row.split_strategy:<10} | {dummy_strategy:<7} | {row.subset:<10} | "
          f"{row.n_structures:>5,} | {row.mae_pki:>7.3f} | {row.rmse_pki:>7.3f} | {r2_text:>7}")

Protocol   | Dummy   | Subset     |     N |     MAE |    RMSE |      R²
random     | mean    | train      | 2,860 |   0.960 |   1.151 |   0.000
random     | mean    | validation |   358 |   0.972 |   1.189 |  -0.001
random     | median  | train      | 2,860 |   0.959 |   1.151 |  -0.001
random     | median  | validation |   358 |   0.971 |   1.188 |  -0.000
scaffold   | mean    | train      | 2,778 |   0.959 |   1.152 |   0.000
scaffold   | mean    | validation |   461 |   0.956 |   1.146 |  -0.002
scaffold   | median  | train      | 2,778 |   0.959 |   1.153 |  -0.001
scaffold   | median  | validation |   461 |   0.956 |   1.147 |  -0.004


## 6. Save this dummy-baseline run

A numbered folder under `provenance/models/dummy_baselines` holds all four fitted dummies, their training constants, prediction rows, and metrics. Numeric values are exported at full precision and rounded only in the notebook display. The completion manifest is withheld until the saved files pass readback verification.

In [7]:
dummy_parent = project_root / "provenance/models/dummy_baselines"
# Numbered runs keep previous results intact; dates are recorded in manifest.json.
dummy_parent.mkdir(parents=True, exist_ok=True)
# Start above the highest saved run number after superseded runs are removed.
existing_run_numbers = [
    int(path.name[4:]) for path in dummy_parent.glob("run_*")
    if path.is_dir() and path.name[4:].isdigit()
]
run_number = max(existing_run_numbers, default=0) + 1
while True:
    dummy_run_name = f"run_{run_number:02d}"
    dummy_folder = dummy_parent / dummy_run_name
    try:
        dummy_folder.mkdir(exist_ok=False)
        break
    except FileExistsError:
        run_number += 1
models_path = dummy_folder / "models.joblib.gz"
training_path = dummy_folder / "training_summary.csv"
prediction_path = dummy_folder / "predictions.csv"
metrics_path = dummy_folder / "metrics.csv"
dummy_manifest_path = dummy_folder / "manifest.json"

joblib.dump(models, models_path, compress=("gzip", 3))
training_summary.to_csv(training_path, index=False)
predictions.to_csv(prediction_path, index=False)
metrics.to_csv(metrics_path, index=False)
print(f"Saved four dummies and three reports to {dummy_folder.relative_to(project_root).as_posix()}")
print("The run remains incomplete until saved-file verification passes.")


Saved four dummies and three reports to data/processed/dummy_baselines/run_03
The run remains incomplete until saved-file verification passes.


## 7. Verify models, predictions, and metrics from disk

The saved CSVs are reopened to check keys, numeric arithmetic, observed targets, and independently recalculated metrics. The saved model artifact is reloaded from this run and checked against the training-only constants and pre-serialization predictions. The preparation inputs are hashed again, then the dummy completion manifest is written last.

In [8]:
saved_predictions = pd.read_csv(prediction_path, dtype=str, keep_default_na=False)
saved_metrics = pd.read_csv(metrics_path, dtype=str, keep_default_na=False)
saved_training = pd.read_csv(training_path, dtype=str, keep_default_na=False)
assert list(saved_predictions.columns) == list(predictions.columns)
assert list(saved_metrics.columns) == list(metrics.columns)
assert list(saved_training.columns) == list(training_summary.columns)
assert len(saved_predictions) == len(predictions)
assert len(saved_metrics) == 8 and len(saved_training) == 4
assert not saved_predictions.duplicated(["split_strategy", "model", "subset", "rdkit_smiles"]).any()
assert set(saved_predictions["subset"]) == {"train", "validation"}
target_by_key = dict(zip(structure_keys, y))
for field in ("observed_pki", "predicted_pki", "residual", "absolute_error", "squared_error"):
    saved_predictions[field] = pd.to_numeric(saved_predictions[field], errors="raise")
    assert np.isfinite(saved_predictions[field]).all(), field
np.testing.assert_allclose(saved_predictions["observed_pki"],
                           saved_predictions["rdkit_smiles"].map(target_by_key), rtol=0, atol=1e-12)
rebuilt_residual = saved_predictions["predicted_pki"] - saved_predictions["observed_pki"]
np.testing.assert_allclose(saved_predictions["residual"], rebuilt_residual, rtol=0, atol=1e-12)
np.testing.assert_allclose(saved_predictions["absolute_error"], np.abs(rebuilt_residual), rtol=0, atol=1e-12)
np.testing.assert_allclose(saved_predictions["squared_error"], rebuilt_residual ** 2, rtol=0, atol=1e-12)

# Rebuild coverage and metrics from saved predictions for each fitted model.
for strategy in STRATEGIES:
    for dummy_strategy in DUMMY_STRATEGIES:
        model_name = f"{strategy}_dummy_{dummy_strategy}"
        for subset in ("train", "validation"):
            rows = saved_predictions.loc[
                saved_predictions["split_strategy"].eq(strategy)
                & saved_predictions["model"].eq(model_name)
                & saved_predictions["subset"].eq(subset)
            ]
            assert set(rows["rdkit_smiles"]) == set(structure_keys[split_indices[strategy][subset]])
            assert set(rows["rdkit_smiles"]).isdisjoint(structure_keys[split_indices[strategy]["test"]])
            metric = saved_metrics.loc[
                saved_metrics["split_strategy"].eq(strategy)
                & saved_metrics["model"].eq(model_name)
                & saved_metrics["subset"].eq(subset)
            ]
            assert len(metric) == 1
            metric = metric.iloc[0]
            assert int(metric["n_structures"]) == len(rows)
            errors = rows["predicted_pki"].to_numpy() - rows["observed_pki"].to_numpy()
            assert np.isclose(float(metric["mae_pki"]), np.mean(np.abs(errors)), rtol=0, atol=1e-12)
            assert np.isclose(float(metric["rmse_pki"]), np.sqrt(np.mean(errors ** 2)), rtol=0, atol=1e-12)
            observed = rows["observed_pki"].to_numpy()
            if len(rows) >= 2 and np.max(observed) > np.min(observed):
                expected_r2 = 1 - np.sum(errors ** 2) / np.sum((observed - np.mean(observed)) ** 2)
                assert metric["r2_status"] == "defined"
                assert np.isclose(float(metric["r2"]), expected_r2, rtol=0, atol=1e-12)
            else:
                assert metric["r2_status"] == "undefined_small_or_constant_target" and metric["r2"] == ""

# Reload this run's own four estimators and verify their training-only constants.
loaded_models = joblib.load(models_path)
assert set(loaded_models) == set(models)
assert not saved_training.duplicated(["split_strategy", "model"]).any()
for strategy in STRATEGIES:
    for dummy_strategy in DUMMY_STRATEGIES:
        name = f"{strategy}_dummy_{dummy_strategy}"
        loaded = loaded_models[name]
        assert isinstance(loaded, DummyRegressor)
        assert loaded.strategy == dummy_strategy and loaded.n_features_in_ == feature_width
        np.testing.assert_allclose(
            loaded.predict(X[training_slices[strategy]]),
            models[name].predict(X[training_slices[strategy]]), rtol=0, atol=1e-12,
        )
        np.testing.assert_allclose(loaded.predict(X[training_slices[strategy]]),
                                   independent_constants[name], rtol=0, atol=1e-10)
        row = saved_training.loc[saved_training["model"].eq(name)].iloc[0]
        assert row["split_strategy"] == strategy and row["dummy_strategy"] == dummy_strategy
        assert int(row["training_structures"]) == len(split_indices[strategy]["train"])
        assert int(row["features"]) == feature_width
        assert np.isclose(float(row["training_constant_pki"]), independent_constants[name], rtol=0, atol=1e-10)
        assert np.isfinite(float(row["fit_seconds"]))

assert sha256_file(prep_manifest_path) == source_manifest_hash
for filename, path in artifact_paths.items():
    relative_name = (prep_folder_rel / filename).as_posix()
    assert sha256_file(path) == prep_manifest["artifact_sha256"][relative_name], filename

# The completion record binds this notebook's fitted models and outputs to the same frozen split handoff.
dummy_manifest = {
    "run_started_at_utc": run_started_at_utc,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "notebooks/05_dummy_baselines.ipynb",
    "stage": "dummy_training_and_evaluation",
    "output_folder": dummy_folder.relative_to(project_root).as_posix(),
    "source_preparation_manifest": PREP_MANIFEST_REL.as_posix(),
    "source_preparation_manifest_sha256": source_manifest_hash,
    "source_artifact_sha256": {
        (prep_folder_rel / filename).as_posix(): prep_manifest["artifact_sha256"][
            (prep_folder_rel / filename).as_posix()
        ] for filename in required_filenames
    },
    "model_names": sorted(loaded_models),
    "model_parameters": {name: loaded.get_params() for name, loaded in loaded_models.items()},
    "training_summary": training_summary.to_dict("records"),
    "counts": {
        "fitted_dummies": len(loaded_models), "predictions": len(saved_predictions),
        "metric_rows": len(saved_metrics),
        "subsets": {strategy: {
            subset: len(split_indices[strategy][subset]) for subset in SUBSETS
        } for strategy in STRATEGIES},
    },
    "metric_definitions": {
        "residual": "predicted_pki - observed_pki",
        "mae_pki": "mean absolute residual in pKi units",
        "rmse_pki": "square root of mean squared residual in pKi units",
        "r2": "1 - sum(squared residual) / sum((observed - evaluated-subset mean)^2); undefined if n<2 or target constant",
    },
    "output_sha256": {
        path.relative_to(project_root).as_posix(): sha256_file(path)
        for path in (models_path, training_path, prediction_path, metrics_path)
    },
    "verification": {
        "input_hashes": True, "saved_models_and_training_constants": True,
        "prediction_keys_and_arithmetic": True, "metrics_recomputed": True,
        "test_excluded": True,
    },
    "fitting_performed": True,
    "tuning_performed": False,
    "test_predictions_performed": False,
    "test_evaluation_performed": False,
    "software_versions": installed_versions,
}
assert not dummy_manifest_path.exists()
with dummy_manifest_path.open("w", encoding="utf-8") as stream:
    json.dump(dummy_manifest, stream, indent=2)
print(f"Verified four fitted dummies, {len(saved_predictions):,} prediction rows, and eight metric rows.")
print(f"Completed dummy run: {dummy_folder.relative_to(project_root).as_posix()}")

Verified four fitted dummies, 12,914 prediction rows, and eight metric rows.
Completed dummy run: data/processed/dummy_baselines/run_03


## 8. Select the dummy strategy using training-only cross-validation

Each protocol keeps its frozen training, validation, and test sets. Five-fold cross-validation (CV) subdivides **only training rows**: four folds fit each candidate and the remaining fold scores it. The random protocol uses shuffled KFold; the scaffold protocol uses GroupKFold so a scaffold cannot appear on both sides of an internal fold. The same fold recipe is used in both model notebooks.

Choose the candidate with the lowest mean fold MAE, then refit it on all original training rows. Outer validation compares the selected model with the baseline afterward. Its scores do not choose the search winner. The reserved test set remains unused until model choices and the final refit procedure are settled. These are the best settings **within this declared grid**, not a guarantee of a global optimum.

The frozen random protocol can place identical fingerprints across folds; the scaffold protocol keeps scaffolds together but can still have fingerprint collisions across different scaffolds. This search preserves the preparation policy and its documented similarity limitations.

Documentation: [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), [GroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html).

Mean and median dummies have no tree/network complexity to tune. Their equivalent comparison selects the constant-prediction strategy; both fixed baselines remain reported. With MAE as the objective, median is a natural candidate, but the saved CV results decide the selection.

In [9]:
from sklearn.model_selection import GridSearchCV, KFold, GroupKFold, ParameterGrid

# Freeze the search design before fitting candidates or inspecting their validation scores.
tuning_started_at_utc = datetime.now(timezone.utc).isoformat()
tuning_protocols = ("random", "scaffold")
tuning_n_folds = 5
tuning_cv_splits = {}
tuning_fold_rows = []
for protocol in tuning_protocols:
    train_indices = split_indices[protocol]["train"]
    train_keys = structure_keys[train_indices]
    scaffold_by_key = assignments.loc[
        assignments["split_strategy"].eq(protocol)
    ].set_index("rdkit_smiles")["scaffold"]
    train_groups = scaffold_by_key.loc[train_keys].to_numpy()
    # These indices address only the training array; outer validation/test never enter CV.
    if protocol == "random":
        splitter = KFold(n_splits=tuning_n_folds, shuffle=True, random_state=42)
        folds = list(splitter.split(train_indices))
    else:
        splitter = GroupKFold(n_splits=tuning_n_folds)
        folds = list(splitter.split(train_indices, groups=train_groups))
    heldout_counts = np.zeros(len(train_indices), dtype=int)
    for fold_number, (fit_local, valid_local) in enumerate(folds, start=1):
        assert set(fit_local).isdisjoint(valid_local)
        assert set(fit_local) | set(valid_local) == set(range(len(train_indices)))
        if protocol == "scaffold":
            assert set(train_groups[fit_local]).isdisjoint(train_groups[valid_local])
        heldout_counts[valid_local] += 1
        for local_index in valid_local:
            tuning_fold_rows.append({
                "split_strategy": protocol, "rdkit_smiles": train_keys[local_index],
                "scaffold": train_groups[local_index], "cv_validation_fold": fold_number,
            })
    assert np.all(heldout_counts == 1)
    tuning_cv_splits[protocol] = folds

tuning_folds = pd.DataFrame(tuning_fold_rows)
# Mean and median are the only choices here: this is strategy selection for constants.
# Each candidate learns its own constant afresh inside every training fold.
tuning_grid = {"strategy": ["mean", "median"]}
tuning_estimator = DummyRegressor()
tuning_baseline_manifest_hash = sha256_file(dummy_manifest_path)
print(f"Candidates per protocol: {len(list(ParameterGrid(tuning_grid)))}; CV folds: {tuning_n_folds}")

Candidates per protocol: 2; CV folds: 5


### Run the declared search

The search fits every candidate on the saved training folds and refits the winner. Fold scores below are internal CV validation scores, not scores on the reserved test set.

In [10]:
# Search candidates sequentially; each forest may parallelize its own trees.
# Negative MAE follows sklearn's larger-is-better convention; reports use positive MAE.
tuning_models = {}
tuning_searches = {}
tuning_result_rows = []
tuning_selection = {}
for protocol in tuning_protocols:
    train_indices = split_indices[protocol]["train"]
    search = GridSearchCV(
        tuning_estimator, tuning_grid, scoring="neg_mean_absolute_error",
        cv=tuning_cv_splits[protocol], refit=True, n_jobs=1,
        error_score="raise", return_train_score=False,
    )
    started = perf_counter()
    search.fit(X[train_indices], y[train_indices])
    elapsed = perf_counter() - started
    # refit=True fits the CV winner on all supplied training rows, with no outer holdout.
    tuning_models[protocol] = search.best_estimator_
    tuning_searches[protocol] = search
    tuning_selection[protocol] = {
        "best_candidate_index": int(search.best_index_),
        "best_parameters": search.best_params_,
        "best_mean_cv_mae_pki": float(-search.best_score_),
        "effective_parameters": search.best_estimator_.get_params(),
        "training_structures": int(len(train_indices)), "search_seconds": elapsed,
        "refit_seconds": float(search.refit_time_),
    }
    for candidate_index, parameters in enumerate(search.cv_results_["params"]):
        result = {
            "split_strategy": protocol, "candidate_index": candidate_index,
            "parameters_json": json.dumps(parameters, sort_keys=True),
            "mean_cv_mae_pki": float(-search.cv_results_["mean_test_score"][candidate_index]),
            "std_cv_mae_pki": float(search.cv_results_["std_test_score"][candidate_index]),
            "rank": int(search.cv_results_["rank_test_score"][candidate_index]),
            "selected": candidate_index == search.best_index_,
        }
        # sklearn calls its internal held-out folds 'test'; these are training-only CV folds.
        for fold in range(tuning_n_folds):
            result[f"fold_{fold + 1}_validation_mae_pki"] = float(
                -search.cv_results_[f"split{fold}_test_score"][candidate_index]
            )
        tuning_result_rows.append(result)
    print(f"{protocol:<10} | best CV MAE {(-search.best_score_):.4f} | {elapsed:.1f} seconds")
    print(f"Selected parameters: {search.best_params_}")
tuning_results = pd.DataFrame(tuning_result_rows)

random     | best CV MAE 0.9599 | 0.0 seconds
Selected parameters: {'strategy': 'mean'}
scaffold   | best CV MAE 0.9604 | 0.0 seconds
Selected parameters: {'strategy': 'median'}


### Compare baseline and selected-model predictions

Training scores describe fit; outer validation scores describe generalization. Optimization can improve CV without improving outer validation. Retain both results and avoid repeatedly changing the grid in response to this holdout.

In [11]:
# Candidate selection is finished before predicting outer validation structures.
tuning_prediction_rows = []
tuning_metric_rows = []
for protocol in tuning_protocols:
    for subset in ("train", "validation"):
        indices = split_indices[protocol][subset]
        predicted = tuning_models[protocol].predict(X[indices])
        observed = y[indices]
        assert np.isfinite(predicted).all()
        for key, actual, estimate in zip(structure_keys[indices], observed, predicted):
            residual = float(estimate - actual)
            tuning_prediction_rows.append({
                "split_strategy": protocol, "subset": subset, "rdkit_smiles": key,
                "observed_pki": float(actual), "predicted_pki": float(estimate),
                "residual": residual, "absolute_error": abs(residual),
                "squared_error": residual ** 2,
            })
        r2_defined = len(observed) >= 2 and np.max(observed) > np.min(observed)
        tuning_metric_rows.append({
            "split_strategy": protocol, "subset": subset, "n_structures": len(indices),
            "mae_pki": float(mean_absolute_error(observed, predicted)),
            "rmse_pki": float(np.sqrt(mean_squared_error(observed, predicted))),
            "r2": float(r2_score(observed, predicted)) if r2_defined else np.nan,
            "r2_status": "defined" if r2_defined else "undefined_small_or_constant_target",
        })
tuning_predictions = pd.DataFrame(tuning_prediction_rows)
tuning_metrics = pd.DataFrame(tuning_metric_rows)

# Keep every baseline in the comparison, even when search does not improve validation.
tuning_baseline_metrics = metrics.copy()
tuning_baseline_metrics["model_variant"] = "dummy_" + tuning_baseline_metrics["model"].str.rsplit("_", n=1).str[-1]
tuning_baseline_metrics = tuning_baseline_metrics.drop(columns="model")
tuning_selected_metrics = tuning_metrics.assign(model_variant="cv_selected")
tuning_comparison = pd.concat([tuning_baseline_metrics, tuning_selected_metrics], ignore_index=True)
print(f"{'Protocol':<10} | {'Variant':<16} | {'Val. MAE':>9} | {'Val. RMSE':>9} | {'Val. R²':>9}")
for row in tuning_comparison.loc[tuning_comparison["subset"].eq("validation")].itertuples():
    print(f"{row.split_strategy:<10} | {row.model_variant:<16} | {row.mae_pki:>9.4f} | "
          f"{row.rmse_pki:>9.4f} | {row.r2:>9.4f}")
print("CV selected the parameters. Outer validation is a comparison; reserved test remains unused.")
print(f"\n{'Protocol':<10} | {'Observed':>9} | {'Predicted':>9} | {'Abs. error':>10}")
for row in tuning_predictions.loc[tuning_predictions["subset"].eq("validation")].groupby("split_strategy").head(2).itertuples():
    print(f"{row.split_strategy:<10} | {row.observed_pki:>9.4f} | {row.predicted_pki:>9.4f} | {row.absolute_error:>10.4f}")

Protocol   | Variant          |  Val. MAE | Val. RMSE |   Val. R²
random     | dummy_mean       |    0.9725 |    1.1890 |   -0.0015
random     | dummy_median     |    0.9710 |    1.1883 |   -0.0003
scaffold   | dummy_mean       |    0.9560 |    1.1459 |   -0.0016
scaffold   | dummy_median     |    0.9558 |    1.1473 |   -0.0041
random     | cv_selected      |    0.9725 |    1.1890 |   -0.0015
scaffold   | cv_selected      |    0.9558 |    1.1473 |   -0.0041
CV selected the parameters. Outer validation is a comparison; reserved test remains unused.

Protocol   |  Observed | Predicted | Abs. error
random     |    5.7447 |    7.0970 |     1.3523
random     |    7.6198 |    7.0970 |     0.5228
scaffold   |    6.3757 |    7.0362 |     0.6605
scaffold   |    5.4527 |    7.0362 |     1.5835


### Save the search and its provenance

The baseline run receives a `tuning` subfolder containing seven artifacts: fold assignments, all candidate scores, selected parameters, two refitted models, predictions, metrics, and a baseline comparison. A final eighth file, the completion manifest, is written only after readback checks pass.

In [12]:
# A new child folder links this search to the baseline from the same notebook execution.
tuning_folder = dummy_folder / "tuning"
tuning_folder.mkdir(exist_ok=False)
tuning_tables = {
    "cv_folds.csv": tuning_folds,
    "cv_results.csv": tuning_results,
    "predictions.csv": tuning_predictions,
    "metrics.csv": tuning_metrics,
    "baseline_comparison.csv": tuning_comparison,
}
for filename, table in tuning_tables.items():
    table.to_csv(tuning_folder / filename, index=False)
joblib.dump(tuning_models, tuning_folder / "models.joblib.gz", compress=("gzip", 3))
with (tuning_folder / "best_parameters.json").open("w", encoding="utf-8") as stream:
    json.dump(tuning_selection, stream, indent=2)
print(f"Search artifacts: {tuning_folder.relative_to(project_root).as_posix()}")

Search artifacts: data/processed/dummy_baselines/run_03/tuning


### Verify saved search artifacts

Check fold membership, candidate selection, all saved model predictions, metric arithmetic, and source hashes. Every prediction is checked against its own protocol to exclude that protocol’s reserved test structures.

In [13]:
# Read the files back before marking the search complete.
for filename, original in tuning_tables.items():
    reloaded = pd.read_csv(tuning_folder / filename, keep_default_na=False)
    expected = original.fillna("")
    pd.testing.assert_frame_equal(reloaded, expected, check_dtype=False, rtol=1e-12, atol=1e-12)
checked_folds = pd.read_csv(tuning_folder / "cv_folds.csv", keep_default_na=False)
checked_results = pd.read_csv(tuning_folder / "cv_results.csv")
checked_predictions = pd.read_csv(tuning_folder / "predictions.csv")
checked_metrics = pd.read_csv(tuning_folder / "metrics.csv")
checked_models = joblib.load(tuning_folder / "models.joblib.gz")
with (tuning_folder / "best_parameters.json").open(encoding="utf-8") as stream:
    checked_selection = json.load(stream)
assert checked_selection == tuning_selection
assert set(checked_models) == set(tuning_protocols)
assert not checked_folds.duplicated(["split_strategy", "rdkit_smiles"]).any()
assert not checked_predictions.duplicated(["split_strategy", "subset", "rdkit_smiles"]).any()
assert set(checked_predictions["subset"]) == {"train", "validation"}

for protocol in tuning_protocols:
    train_indices = split_indices[protocol]["train"]
    fold_rows = checked_folds.loc[checked_folds["split_strategy"].eq(protocol)]
    assert set(fold_rows["rdkit_smiles"]) == set(structure_keys[train_indices])
    assert set(fold_rows["cv_validation_fold"]) == set(range(1, tuning_n_folds + 1))
    if protocol == "scaffold":
        assert fold_rows.groupby("scaffold")["cv_validation_fold"].nunique().eq(1).all()
    candidate_rows = checked_results.loc[checked_results["split_strategy"].eq(protocol)]
    assert len(candidate_rows) == len(list(ParameterGrid(tuning_grid)))
    fold_scores = candidate_rows[[f"fold_{fold}_validation_mae_pki" for fold in range(1, tuning_n_folds + 1)]].to_numpy()
    assert np.isfinite(fold_scores).all() and (fold_scores >= 0).all()
    np.testing.assert_allclose(candidate_rows["mean_cv_mae_pki"], fold_scores.mean(axis=1), atol=1e-12)
    np.testing.assert_allclose(candidate_rows["std_cv_mae_pki"], fold_scores.std(axis=1), atol=1e-12)
    selected = candidate_rows.loc[candidate_rows["selected"]]
    assert len(selected) == 1 and selected.iloc[0]["rank"] == 1
    assert selected.iloc[0]["candidate_index"] == checked_selection[protocol]["best_candidate_index"]
    assert np.isclose(selected.iloc[0]["mean_cv_mae_pki"], candidate_rows["mean_cv_mae_pki"].min(), atol=1e-12)
    assert json.loads(selected.iloc[0]["parameters_json"]) == checked_selection[protocol]["best_parameters"]
    assert checked_models[protocol].get_params() == tuning_models[protocol].get_params()
    assert checked_models[protocol].n_features_in_ == feature_width
    for subset in ("train", "validation"):
        indices = split_indices[protocol][subset]
        rows = checked_predictions.loc[
            checked_predictions["split_strategy"].eq(protocol) & checked_predictions["subset"].eq(subset)
        ].set_index("rdkit_smiles")
        assert set(rows.index) == set(structure_keys[indices])
        assert set(rows.index).isdisjoint(structure_keys[split_indices[protocol]["test"]])
        rows = rows.loc[structure_keys[indices]]
        # Full readback predictions verify the serialized winner as well as CSV alignment.
        np.testing.assert_allclose(rows["predicted_pki"], checked_models[protocol].predict(X[indices]), rtol=0, atol=1e-10)
        np.testing.assert_allclose(rows["observed_pki"], y[indices], rtol=0, atol=1e-12)
        errors = rows["predicted_pki"].to_numpy() - y[indices]
        np.testing.assert_allclose(rows["residual"], errors, rtol=0, atol=1e-12)
        np.testing.assert_allclose(rows["absolute_error"], np.abs(errors), rtol=0, atol=1e-12)
        np.testing.assert_allclose(rows["squared_error"], errors ** 2, rtol=0, atol=1e-12)
        metric = checked_metrics.loc[
            checked_metrics["split_strategy"].eq(protocol) & checked_metrics["subset"].eq(subset)
        ].iloc[0]
        assert metric["n_structures"] == len(indices)
        np.testing.assert_allclose(metric["mae_pki"], np.abs(errors).mean(), atol=1e-12)
        np.testing.assert_allclose(metric["rmse_pki"], np.sqrt((errors ** 2).mean()), atol=1e-12)
        if metric["r2_status"] == "defined":
            expected_r2 = 1 - np.sum(errors ** 2) / np.sum((y[indices] - y[indices].mean()) ** 2)
            np.testing.assert_allclose(metric["r2"], expected_r2, atol=1e-12)

# Link the completed search to frozen data and the completed baseline run.
assert sha256_file(prep_manifest_path) == source_manifest_hash
for filename, path in artifact_paths.items():
    assert sha256_file(path) == prep_manifest["artifact_sha256"][(prep_folder_rel / filename).as_posix()]
assert sha256_file(dummy_manifest_path) == tuning_baseline_manifest_hash
tuning_output_paths = [tuning_folder / filename for filename in tuning_tables]
tuning_output_paths += [tuning_folder / "models.joblib.gz", tuning_folder / "best_parameters.json"]
tuning_manifest = {
    "stage": 'dummy_strategy_search', "notebook": 'notebooks/05_dummy_baselines.ipynb',
    "run_started_at_utc": tuning_started_at_utc,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "output_folder": tuning_folder.relative_to(project_root).as_posix(),
    "source_preparation_manifest": PREP_MANIFEST_REL.as_posix(),
    "source_preparation_manifest_sha256": source_manifest_hash,
    "source_artifact_sha256": {
        path.relative_to(project_root).as_posix(): sha256_file(path) for path in artifact_paths.values()
    },
    "baseline_manifest": dummy_manifest_path.relative_to(project_root).as_posix(),
    "baseline_manifest_sha256": tuning_baseline_manifest_hash,
    "configuration": {
        "search": "GridSearchCV", "parameter_grid": tuning_grid,
        "scoring": "neg_mean_absolute_error", "refit": True, "search_n_jobs": 1,
        "folds": tuning_n_folds,
        "random_cv": {"method": "KFold", "shuffle": True, "random_state": 42},
        "scaffold_cv": {"method": "GroupKFold", "shuffle": False, "group": "saved_scaffold"},
        "selection_rule": "minimum unweighted mean fold MAE; first candidate breaks ties",
        "scope": "bounded first search; best within the declared grid",
    },
    "selection": tuning_selection,
    "counts": {"candidates_per_protocol": len(list(ParameterGrid(tuning_grid))),
               "selected_models": len(tuning_models), "prediction_rows": len(tuning_predictions)},
    "output_sha256": {path.relative_to(project_root).as_posix(): sha256_file(path) for path in tuning_output_paths},
    "verification": {"input_hashes": True, "saved_tables_and_selection": True,
                     "training_only_cv_folds": True, "scaffold_cv_groups_disjoint": True,
                     "serialized_model_predictions": True, "metrics_recomputed": True, "test_excluded": True},
    "fitting_performed": True, "tuning_performed": True,
    "outer_validation_used_for_search": False,
    "test_predictions_performed": False, "test_evaluation_performed": False,
    "software_versions": installed_versions,
}
with (tuning_folder / "manifest.json").open("x", encoding="utf-8") as stream:
    json.dump(tuning_manifest, stream, indent=2)
print("Verified training-only CV, saved models, predictions, metrics, and source hashes.")
print(f"Completed search: {tuning_folder.relative_to(project_root).as_posix()}")

Verified training-only CV, saved models, predictions, metrics, and source hashes.
Completed search: data/processed/dummy_baselines/run_03/tuning


## Stopping point

This notebook contains the fixed baseline and a completed CV search, with training and validation predictions for both. Use this same sequence for later model families, choosing a suitable declared search space for each. After comparing model families, freeze the final choices and refit policy before performing the reserved test evaluation once. No test predictions have been made here.

## Refresh the project reference
After this stage passes its checks, update the model comparison and project file guide. The presentation workbook stays unchanged. Existing model results remain labeled with the dataset that produced them.


In [ ]:
# Verify saved evidence and refresh Markdown references; preserve the presentation workbook.
from build_project_reference import build_reference
build_reference()
